# ST fit toys
## Do some toys with ST fits

### This was a failed project because FFT didn't work...

### RooFit model

In [ ]:
using namespace RooFit;

In [ ]:
RooRealVar MBC("MBC", "", 1.83, 1.8865);
MBC.setBins(1000);
MBC.setBins(1000, "cache")

In [ ]:
RooRealVar Nsig("Nsig", "", 1174820.0, 0.0, 2.0e6);
RooRealVar Nbkg("Nbkg", "", 664064.0, 0.0, 1.0e6);

In [ ]:
TChain MCChain("KSpipiSingleTag");
std::string Filename("/data/bes3/tat/");
Filename += "KKpipi_StrongPhase_Analysis_4Bins_20fb/Selection/";
Filename += "SignalMC/SingleTag/KSpipi/KSpipi_SingleTag_SignalMC.root";
MCChain.Add(Filename.c_str());
MCChain.SetBranchStatus("*", 0);
MCChain.SetBranchStatus("MBC", 1);
auto MCTree = MCChain.CloneTree(1000);
RooDataSet MCSignal("MCSignal", "", MCTree, RooArgList(MBC));
RooKeysPdf Keys("Keys", "", MBC, MCSignal);

In [ ]:
RooRealVar Mean1("Mean1", "", 0.0, -0.002, 0.002);
RooRealVar Mean2("Mean2", "", 0.0, -0.002, 0.002);
RooRealVar Sigma1("Sigma1", "", 0.00457306, 0.00005, 0.008);
RooRealVar Sigma2("Sigma2", "", 0.000418101, 0.00005, 0.008);
RooRealVar frac("frac", "", 0.5, 0.0, 1.0);

In [ ]:
RooGaussian Gauss1("Gauss1", "", MBC, Mean1, Sigma1);
RooGaussian Gauss2("Gauss2", "", MBC, Mean2, Sigma2);
RooAddPdf Resolution("Resolution", "", Gauss1, Gauss2, frac);

In [ ]:
//RooFFTConvPdf SignalModel("SignalModel", "", MBC, Keys, Resolution);
RooNumConvPdf SignalModel("SignalModel", "", MBC, Keys, Resolution);

In [ ]:
RooRealVar c("c", "", -12.0, -20.0, -0.01);
RooRealVar End("End", "", 1.8865);
RooArgusBG Argus("Argus", "", MBC, End, c);

In [ ]:
RooAddPdf Model("Model", "",
                RooArgList(SignalModel, Argus),
                RooArgList(Nsig, Nbkg));

In [ ]:
auto FitParameters = Model.getParameters(MBC);
auto InitialParams = FitParameters->snapshot();

In [ ]:
FitParameters->Print("V");

In [ ]:
RooFit::RooMsgService::instance().getStream(1).removeTopic(Eval)
auto testData = Model.generateBinned(MBC);

In [ ]:
auto FitResult = Model.fitTo(*testData,
                             PrintEvalErrors(-1),
                             PrintLevel(-1),
                             Save(true));

In [ ]:
FitResult->Print();

In [ ]:
TCanvas c("c", "", 1200, 900);
auto myFrame = MBC.frame();
//testData->plotOn(myFrame, Binning(100));
Model.plotOn(myFrame, Components("SignalModel"));
myFrame->Draw();
c.Draw()

In [ ]:
c.Close();